In [3]:
import os
import json
import pickle
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm
from diskcache import Cache
import torch
from torch_geometric.data import Data


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "data_dir": "../mall",
            "process_label_file": "../mall/label_process.csv",
            "dataset_cache_dir": "../mall/cache",
            "data_chunk_keys": "data_chunk_keys",
            "chunk": "chunk",
            "padding_length": 41,
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
        }
    elif dataset == "train-ticket":
        return {
            "data_dir": "../train-ticket",
            "process_label_file": "../train-ticket/label_process.csv",
            "dataset_cache_dir": "../train-ticket/cache",
            "data_chunk_keys": "data_chunk_keys",
            "chunk": "chunk",
            "padding_length": 41,
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class GraphBuilder:
    """
    作用：
    1. 读取 data_alignment_3.parquet
    2. 从 label_process.csv 建立 trace_id -> fault_id 映射
    3. 按 TraceID 分组构图
    4. 转成 PyG Data
    5. 缓存并返回 data_list
    """

    def __init__(self, config: dict):
        self.config = config
        self.data_dir = config["data_dir"]
        self.process_label_file = config["process_label_file"]
        self.dataset_cache_dir = config["dataset_cache_dir"]
        self.data_chunk_keys = config["data_chunk_keys"]
        self.chunk_prefix = config["chunk"]
        self.padding_length = config["padding_length"]
        self.outer_name = config["outer_name"]

        self.alignment3_path = os.path.join(self.data_dir, "data_alignment_3.parquet")

    def ensure_cache_dir(self) -> None:
        if not os.path.exists(self.dataset_cache_dir):
            os.mkdir(self.dataset_cache_dir)

    @staticmethod
    def parse_list_string(value):
        import ast
        if isinstance(value, str):
            return ast.literal_eval(value)
        return value

    def load_fault_mapping(self) -> Dict[str, int]:
        """
        从 label_process.csv 中构建:
            trace_id -> fault_id
        """
        label_df = pd.read_csv(self.process_label_file)
        trace_id_2_fault_id = {}

        for _, row in label_df.iterrows():
            trace_id_list = self.parse_list_string(row["trace_id_list"])
            fault_id = row["fault_id"]
            for trace_id in trace_id_list:
                trace_id_2_fault_id[trace_id] = fault_id

        return trace_id_2_fault_id

    def load_alignment_df(self) -> pd.DataFrame:
        return pd.read_parquet(self.alignment3_path)

    def create_graph_from_df(self, df: pd.DataFrame) -> nx.DiGraph:
        """
        把单个 trace 对应的 span 表转成 networkx 图
        把一个 trace 的所有 span，构造成一张图（节点=span，边=调用关系，特征=多模态数据）
        """
        G = nx.DiGraph()

        outer_name = self.outer_name.copy()
        if "RpcName" in outer_name:
            outer_name.remove("RpcName")

        # RpcName（向量） logs（向量序列） metrics（时间序列） Duration / ResultCode
        node_attributes = df.columns.difference(outer_name)

        # 对一个trace中每一个span
        for _, row in df.iterrows():
            node_attrs = row[node_attributes].to_dict()

            if "ResultCode" in node_attrs:
                try:
                    node_attrs["ResultCode"] = int(node_attrs["ResultCode"])
                except Exception:
                    pass

            for attr_id in node_attrs:
                if isinstance(node_attrs[attr_id], bytes):
                    node_attrs[attr_id] = np.array(eval(node_attrs[attr_id].decode()))

            # 每个节点 = 一个 span + 所有特征
            G.add_node(row["SpanId"], **node_attrs)

            # 每个节点 = 一个 span + 所有特征 双向边
            if row["ParentSpanId"] != "0":
                G.add_edge(row["ParentSpanId"], row["SpanId"])
                G.add_edge(row["SpanId"], row["ParentSpanId"])

        # 删除空节点
        nodes_to_remove = [node for node, attr in G.nodes(data=True) if len(attr) == 0]
        for node in nodes_to_remove:
            G.remove_node(node)

        return G

    def extract_into_figure(self) -> Tuple[Dict[str, nx.DiGraph], Dict[str, int]]:
        """
        返回：
        - trace_id_2_graphs
        - trace_id_2_fault_id
        """
        # 获取不同traceID对应的错误类型
        '''
            {
                "traceA": 0,
                "traceB": 0,
                "traceC": 1,
                ...
            }
        '''
        trace_id_2_fault_id = self.load_fault_mapping()
        # 得到一个宽表，每一行是一个span对应的数据,将服务接口名RpcName和日志模板用Bert编码为语义向量
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |
        data_df = self.load_alignment_df()

        grouped = data_df.groupby("TraceID")
        # 每个trace_id对应一张图,图节点为span,节点属性为多模态特征
        trace_id_2_graphs = {
            trace_id: self.create_graph_from_df(group)
            for trace_id, group in tqdm(grouped, postfix="build networkx graph")
        }

        return trace_id_2_graphs, trace_id_2_fault_id

    def process_data(
        self,
        trace_id_2_graphs: Dict[str, nx.DiGraph],
        trace_id_2_fault_id: Dict[str, int],
    ) -> List[Data]:
        """
        把 networkx 图转成 PyG Data
        """
        data_list = []

        for trace_id in tqdm(trace_id_2_graphs, postfix="convert to pyg data"):
            G = trace_id_2_graphs[trace_id]

            # 重新编号节点：把原来的 SpanId 映射到 0..N-1
            label_mapping = {label: i for i, label in enumerate(G.nodes())}
            G = nx.relabel_nodes(G, label_mapping)

            nodes_features = []
            drop_flag = False

            for _, node_data in G.nodes(data=True):
                # 1. trace_feature = [Duration, ResultCode, RpcName_embedding]
                trace_feature = torch.tensor(
                    np.concatenate(
                        ([node_data["Duration"]], [node_data["ResultCode"]], node_data["RpcName"])
                    ),
                    dtype=torch.float,
                )

                # 2. logs_feature = [[...], [...], ...]
                # 对一个节点的N条日志,得到的形状为(N, hidden)
                logs_feature = torch.tensor(
                    np.array([l for l in node_data["logs"]]),
                    dtype=torch.float,
                )

                # 3. metrics_feature = 所有 metric 列的时间序列
                # [指标个数, 时间序列长度]
                target_length = self.padding_length

                for metric_name in node_data:
                    if metric_name not in ["Duration", "ResultCode", "RpcName", "logs"]:
                        pad_size = target_length - len(node_data[metric_name])

                        if pad_size > 0:
                            try:
                                node_data[metric_name] = np.pad(
                                    node_data[metric_name],
                                    (0, pad_size),
                                    mode="wrap",
                                )
                            except Exception:
                                drop_flag = True
                                break

                if drop_flag:
                    break

                metrics_feature = torch.tensor(
                    np.array([
                        node_data[metric_name]
                        for metric_name in node_data
                        if metric_name not in ["Duration", "ResultCode", "RpcName", "logs"]
                    ]),
                    dtype=torch.float,
                )

                nodes_features.append(
                    {
                        "trace_feature": trace_feature,
                        "logs_feature": logs_feature,
                        "metrics_feature": metrics_feature,
                    }
                )

            if drop_flag:
                print(f"drop trace id {trace_id}")
                continue

            edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()

            pyg_data = Data(
                x=nodes_features,
                edge_index=edge_index,
                label=trace_id_2_fault_id[trace_id],
                trace_id=trace_id,
                num_nodes=len(nodes_features),
            )
            data_list.append(pyg_data)

        return data_list

    @staticmethod
    def chunk_data(data: List[Data], chunk_size: int):
        return (data[i:i + chunk_size] for i in range(0, len(data), chunk_size))

    def get_data(self, reload: bool = False) -> List[Data]:
        """
        主逻辑：
        - 如果 cache 里没有，则构图并缓存
        - 否则直接从 cache 读
        """
        self.ensure_cache_dir()
        cache = Cache(self.dataset_cache_dir, size_limit=int(9 * 1e10))

        if reload or self.data_chunk_keys not in cache:
            trace_id_2_graphs, trace_id_2_fault_id = self.extract_into_figure()
            data_list = self.process_data(trace_id_2_graphs, trace_id_2_fault_id)

            del trace_id_2_graphs, trace_id_2_fault_id

            # 每 2000 个 graph 存一块
            chunk_size = 2000
            chunk_keys = []

            length = len([i for i in range(0, len(data_list), chunk_size)])
            for i, chunk in tqdm(
                enumerate(self.chunk_data(data_list, chunk_size)),
                total=length,
                postfix="cache chunks",
            ):
                key = f"{self.chunk_prefix}_{i}"
                '''
                    {
                        "chunk_0": bytes,
                        "chunk_1": bytes,
                        ...
                    }
                '''
                cache.set(key, pickle.dumps(chunk))
                chunk_keys.append(key)
            '''
            {
                "chunk_0": pickle(bytes of List[Data]),
                "chunk_1": pickle(bytes of List[Data]),
                ...
                "data_chunk_keys": ["chunk_0", "chunk_1", ...]
            }

            其中
            Data(
                x = {
                    "trace_feature": tensor(Duration, ResultCode, RpcName_embedding), shape=(2+Bert_hidden, )
                    "logs_feature": tensor, shape=(num_logs, hidden)
                    "metrics_feature": tensor, shape=(num_metrics, padding_length)
                },
                
                edge_index = tensor(2, num_edges),
                label = int,
                trace_id = str, "trace_xxx"
                num_nodes = int
            )
            '''
            cache.set(self.data_chunk_keys, chunk_keys)

        else:
            data_list = []
            chunk_keys = cache.get(self.data_chunk_keys)

            for key in tqdm(chunk_keys, postfix="load cache chunks"):
                chunk = pickle.loads(cache.get(key))
                data_list.extend(chunk)

        return data_list

/home/lenovo/miniconda3/envs/Cadrca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
for dataset in ["mall", "train-ticket"]:   # "mall"或 "train-ticket"
    reload = False
    
    config = get_config(dataset)
    builder = GraphBuilder(config)
    data_list = builder.get_data(reload=reload)
    
    print("\nnum graphs:", len(data_list))
    if len(data_list) > 0:
        print(data_list[0])
        print("label:", data_list[0].label)
        print("trace_id:", data_list[0].trace_id)
        print("num_nodes:", data_list[0].num_nodes)

100%|██████████| 26/26 [02:51<00:00,  6.58s/it, cache chunks]



num graphs: 50354
Data(x=[13], edge_index=[2, 24], label=6, trace_id='0002bd9902357d1aad9c9be28abe694e', num_nodes=13)
label: 6
trace_id: 0002bd9902357d1aad9c9be28abe694e
num_nodes: 13


100%|██████████| 2/2 [00:04<00:00,  2.16s/it, cache chunks]



num graphs: 2522
Data(x=[11], edge_index=[2, 20], label=6, trace_id='00015b3d2e42cce1ec2d01878d812e55', num_nodes=11)
label: 6
trace_id: 00015b3d2e42cce1ec2d01878d812e55
num_nodes: 11
